# 🧠 MTM V5 — Motor Propio (Scanner Independiente)

Este notebook genera automáticamente una **Watchlist propia** sin depender del CDI.

**Pasos:**
1. Scanea **Finviz** para obtener universo con momentum
2. Descarga datos de **Yahoo Finance** (SMA, ATR, RSI)
3. Calcula **Score V4**
4. Filtra por riesgo
5. Escribe el resultado en tu **Google Sheet**

> ⏱️ Tiempo estimado: ~5-8 minutos para 200 tickers


## Paso 0: Instalar librerías necesarias

In [ ]:
!pip install yfinance gspread pandas numpy -q
print('✅ Librerías instaladas')

## Paso 1: Conectar con Google Sheets (Autenticación simple)

Ejecutá esta celda y hacé clic en el **link que aparece** para autorizar con tu cuenta de Google (la misma que usa tu Tracker V4).

In [ ]:
from google.colab import auth
import gspread
from google.auth import default

# Autenticación con TU cuenta de Google (la misma que tiene el Sheet)
auth.authenticate_user()

# Obtener credenciales autenticadas y crear cliente de gspread
creds, _ = default()
gc = gspread.authorize(creds)

print('✅ Autenticado con tu cuenta de Google')

## Paso 2: Abrir tu Spreadsheet

Copiá el **ID** de tu Google Sheet desde la URL:
`https://docs.google.com/spreadsheets/d/ACÁ_ESTÁ_EL_ID/edit`

In [ ]:
SPREADSHEET_ID = 'TU_SPREADSHEET_ID_AQUI'  # ← Pegá el ID de tu Tracker V4

try:
    ss = gc.open_by_key(SPREADSHEET_ID)
    print('✅ Spreadsheet abierto:', ss.title)
    print('Hojas disponibles:', [ws.title for ws in ss.worksheets()])
except Exception as e:
    print('❌ Error:', e)
    print('Verificá que el SPREADSHEET_ID sea correcto.')

## Paso 3: Obtener universo de tickers

**Opción A (Recomendada):** Lista fija S&P 500 desde Wikipedia — más estable, no depende de Finviz.
**Opción B:** Intentar Finviz (puede fallar desde Colab por bloqueo anti-bot).

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time

# ── OPCIÓN A: S&P 500 desde GitHub Datasets (más estable, sin bloqueo) ──
print('📥 Descargando S&P 500 desde dataset público...')
try:
    csv_url = 'https://raw.githubusercontent.com/datasets/s-and-p-500-companies/master/data/constituents.csv'
    sp500_df = pd.read_csv(csv_url)
    universo = sp500_df['Symbol'].str.upper().tolist()
    print(f'✅ S&P 500: {len(universo)} tickers cargados desde CSV')
    print('Primeros 10:', universo[:10])
except Exception as e:
    print(f'⚠️ Error CSV: {e}')
    universo = []

# ── OPCIÓN B: NASDAQ-100 desde Wikipedia (si el CSV falla) ──
if len(universo) == 0:
    try:
        print('📥 Intentando NASDAQ-100 desde Wikipedia...')
        wiki_url = 'https://en.wikipedia.org/wiki/NASDAQ-100'
        tables = pd.read_html(wiki_url)
        # Buscar la tabla que tiene la columna Ticker/Symbol
        for table in tables:
            if 'Ticker' in table.columns or 'Symbol' in table.columns:
                col = 'Ticker' if 'Ticker' in table.columns else 'Symbol'
                universo = table[col].str.upper().tolist()
                break
        print(f'✅ NASDAQ-100: {len(universo)} tickers cargados')
    except Exception as e:
        print(f'⚠️ Error Wikipedia: {e}')
        universo = []

# ── OPCIÓN C: Intentar Finviz (raro que funcione desde Colab) ──
if len(universo) == 0:
    print('🔍 Intentando Finviz como fallback...')
    HEADERS = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        'Accept': 'text/html,application/xhtml+xml',
        'Accept-Language': 'en-US,en;q=0.9'
    }
    
    def scan_finviz(filter_url, max_pages=5):
        tickers = []
        for page in range(1, max_pages + 1):
            url = f"{filter_url}&r={((page-1)*20)+1}"
            try:
                r = requests.get(url, headers=HEADERS, timeout=15)
                if r.status_code != 200:
                    break
                soup = BeautifulSoup(r.text, 'html.parser')
                rows = soup.select('.styled-row')
                if len(rows) <= 1:
                    break
                for row in rows[1:]:
                    ticker_el = row.select_one('.screener-link-primary')
                    if ticker_el:
                        tickers.append(ticker_el.text.strip().upper())
                time.sleep(0.8)
            except:
                break
        return tickers
    
    FILTRO_MOMENTUM = 'https://finviz.com/screener.ashx?v=111&f=cap_midover,ta_perf_1w20o,ta_sma50_pa&ft=4'
    universo = scan_finviz(FILTRO_MOMENTUM, max_pages=10)
    print(f'Finviz: {len(universo)} tickers encontrados')

# ── OPCIÓN D: Lista de respaldo ──
if len(universo) == 0:
    print('⚠️ Usando lista de respaldo de 50 tickers líquidos...')
    universo = ['AAPL','MSFT','NVDA','AMZN','GOOGL','META','TSLA','AVGO','BRK.B','WMT',
                'JPM','V','MA','UNH','HD','PG','JNJ','BAC','ABBV','KO',
                'MRK','PEP','TMO','COST','CVX','ABT','MCD','ADBE','CRM','ACN',
                'WFC','CSCO','TXN','VZ','AMGN','NKE','IBM','QCOM','NEE','PM',
                'RTX','SPGI','LOW','UNP','HON','INTU','CAT','GS','SBUX','BLK']
    print(f'✅ Lista de respaldo: {len(universo)} tickers')

print(f'\n🎯 Universo total: {len(universo)} tickers')

## Paso 4: Descargar datos de Yahoo Finance

Para cada ticker, obtenemos:
- Precio actual, SMA 20/50/200
- ATH y distancia
- ATR(14)
- Beta, RSI (aproximado)

> ⏱️ Esto toma ~3-5 minutos para 200 tickers.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime

def calcular_atr(df, periodo=14):
    """Calcula ATR(14)"""
    high_low = df['High'] - df['Low']
    high_close = np.abs(df['High'] - df['Close'].shift())
    low_close = np.abs(df['Low'] - df['Close'].shift())
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    atr = tr.rolling(window=periodo).mean().iloc[-1]
    return atr

def analizar_ticker(ticker):
    """Analiza un ticker y devuelve dict con métricas."""
    try:
        t = yf.Ticker(ticker)
        hist = t.history(period='1y')
        if len(hist) < 200:
            return None
        
        closes = hist['Close']
        precio = closes.iloc[-1]
        sma20 = closes.rolling(20).mean().iloc[-1]
        sma50 = closes.rolling(50).mean().iloc[-1]
        sma200 = closes.rolling(200).mean().iloc[-1]
        ath = closes.max()
        dist_ath = (ath - precio) / ath
        
        # Performance semanal/mensual/trimestral
        perf_week = (closes.iloc[-1] / closes.iloc[-6] - 1) * 100 if len(closes) >= 6 else 0
        perf_month = (closes.iloc[-1] / closes.iloc[-22] - 1) * 100 if len(closes) >= 22 else 0
        perf_quart = (closes.iloc[-1] / closes.iloc[-66] - 1) * 100 if len(closes) >= 66 else 0
        
        # ATR
        atr = calcular_atr(hist)
        atr_pct = (atr / precio) * 100 if precio > 0 else 0
        
        # RSI simple (aproximado)
        delta = closes.diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=14).mean().iloc[-1]
        loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean().iloc[-1]
        rs = gain / loss if loss > 0 else 0
        rsi = 100 - (100 / (1 + rs)) if rs > 0 else 50
        
        # Info del ticker (beta, sector, etc.)
        info = t.info
        beta = info.get('beta', None)
        empresa = info.get('longName', '') or info.get('shortName', ticker)
        sector = info.get('sector', '')
        
        return {
            'ticker': ticker,
            'empresa': empresa,
            'sector': sector,
            'precio': precio,
            'sma20': sma20,
            'sma50': sma50,
            'sma200': sma200,
            'ath': ath,
            'dist_ath': dist_ath,
            'perf_week': perf_week,
            'perf_month': perf_month,
            'perf_quart': perf_quart,
            'atr': atr,
            'atr_pct': atr_pct,
            'rsi': rsi,
            'beta': beta,
        }
    except Exception as e:
        return None

# Analizar tickers (con progreso)
print(f'📊 Analizando {len(universo)} tickers...')
resultados = []
for i, tk in enumerate(universo):
    res = analizar_ticker(tk)
    if res:
        resultados.append(res)
    if (i + 1) % 50 == 0:
        print(f'  ...{i+1}/{len(universo)} analizados')
    time.sleep(0.05)

print(f'✅ {len(resultados)} tickers analizados con datos completos')

## Paso 5: Calcular Score V4

Aplicamos la misma fórmula del Tracker V4:
- Momentum (35%): Perf Semana/Mes/Trimestre
- Fuerza Relativa (25%): Distancia ATH
- Tendencia (25%): SMA 20/50/200
- Riesgo (15%): RSI, Beta, ATR

In [ ]:
def calcular_score_v4(row):
    """Calcula Score V4 para un ticker."""
    momentum = 0
    pW = row['perf_week']
    pM = row['perf_month']
    pQ = row['perf_quart']
    
    if pW > 20: momentum += 4
    elif pW > 15: momentum += 3
    elif pW > 10: momentum += 2
    elif pW > 5: momentum += 1
    elif pW > 0: momentum += 0.5
    else: momentum -= 1
    
    if pM > 20: momentum += 2
    elif pM > 10: momentum += 1.5
    elif pM > 0: momentum += 0.5
    else: momentum -= 0.5
    
    if pQ > 30: momentum += 1
    elif pQ > 15: momentum += 0.5
    
    # Fuerza relativa (distancia ATH invertida: más cerca = más fuerte)
    fuerza = 0
    if row['dist_ath'] is not None:
        if row['dist_ath'] < 0.05: fuerza += 5
        elif row['dist_ath'] < 0.10: fuerza += 4
        elif row['dist_ath'] < 0.15: fuerza += 3
        elif row['dist_ath'] < 0.25: fuerza += 2
        elif row['dist_ath'] < 0.50: fuerza += 1
    
    # Tendencia (SMAs)
    tendencia = 0
    precio = row['precio']
    if precio > row['sma200']:
        tendencia += 1.5
    if precio > row['sma50']:
        tendencia += 1.0
    if precio > row['sma20']:
        tendencia += 0.5
    
    # Riesgo
    riesgo = 0
    if row['rsi']:
        if row['rsi'] > 80: riesgo -= 1.5
        elif row['rsi'] > 70: riesgo -= 0.5
        elif row['rsi'] >= 50: riesgo += 1.5
        elif row['rsi'] >= 40: riesgo += 0.5
        else: riesgo -= 0.5
    
    if row['beta']:
        if row['beta'] > 2.5: riesgo -= 1.0
        elif row['beta'] > 1.5: riesgo -= 0.5
        elif row['beta'] >= 0.8: riesgo += 0.5
        else: riesgo += 1.0
    
    # ATR/LOW simulado (verde si ATR% > 3%)
    if row['atr_pct'] > 3:
        riesgo += 1
    elif row['atr_pct'] < 1.5:
        riesgo -= 1
    
    score = (momentum * 0.35) + (fuerza * 0.25) + (tendencia * 0.25) + (riesgo * 0.15)
    return round(max(0, min(10, score)), 2)

# Crear DataFrame
df = pd.DataFrame(resultados)
df['score_v4'] = df.apply(calcular_score_v4, axis=1)
df = df.sort_values('score_v4', ascending=False)

print(f'✅ Score calculado para {len(df)} tickers')
print('\n🏆 Top 10:')
print(df[['ticker', 'empresa', 'score_v4', 'perf_week', 'dist_ath']].head(10).to_string(index=False))

## Paso 6: Filtrar por Riesgo

Descartamos tickers que no cumplan:
- Precio > SMA200 (tendencia alcista)
- ATR% > 1.5% (se mueve lo suficiente)
- Score >= 2.0

In [ ]:
UMBRAL_SCORE = 2.0
UMBRAL_ALTA = 4.5
UMBRAL_MEDIA = 3.0

filtrados = df[
    (df['precio'] > df['sma200']) &
    (df['atr_pct'] > 1.5) &
    (df['score_v4'] >= UMBRAL_SCORE)
]

print(f'🎯 Filtrados: {len(filtrados)} tickers (de {len(df)} originales)')
print(f'   🟢 Alta Confianza (≥{UMBRAL_ALTA}): {len(filtrados[filtrados["score_v4"] >= UMBRAL_ALTA])}')
print(f'   🟡 Media Confianza ({UMBRAL_MEDIA}-{UMBRAL_ALTA}): {len(filtrados[(filtrados["score_v4"] >= UMBRAL_MEDIA) & (filtrados["score_v4"] < UMBRAL_ALTA)])}')
print(f'   🟠 Base ({UMBRAL_SCORE}-{UMBRAL_MEDIA}): {len(filtrados[(filtrados["score_v4"] >= UMBRAL_SCORE) & (filtrados["score_v4"] < UMBRAL_MEDIA)])}')

print('\n📋 Candidatas finales:')
print(filtrados[['ticker', 'empresa', 'score_v4', 'sector']].head(20).to_string(index=False))

## Paso 7: Escribir en Google Sheets

Crea/actualiza la hoja **📋 WL V5 Generado** con el mismo formato que el WL CDI.

> 💡 **Recordá:** Tenés que estar autenticado en el Paso 1 para que esto funcione.

In [ ]:
SHEET_WL_V5 = '📋 WL V5 Generado'

# Preparar datos en formato del WL CDI actual
wl_data = []
for _, row in filtrados.iterrows():
    # ATR/LOW simulado: verde si ATR% > 3%
    atr_signal = '🟢 ATR OK' if row['atr_pct'] > 3 else '🟡 ATR Medio' if row['atr_pct'] > 2 else '🔴 ATR Bajo'
    
    wl_data.append([
        row['ticker'],
        row['empresa'],
        row['sector'],
        atr_signal,
        '',  # Earnings (no disponible sin API paga)
        f"{row['perf_week']:.1f}%",
        f"{row['perf_month']:.1f}%",
        '',  # SCTR (no disponible sin StockCharts)
        f"{row['rsi']:.0f}",
        '',  # ADX (no disponible sin datos intradía)
        f"{row['beta']:.2f}" if row['beta'] else '',
        f"{row['atr']:.2f}",
        f"{row['sma20']:.2f}",
    ])

# Headers del WL CDI
headers = ['Ticker', 'Empresa', 'Sector', 'ATR/LOW', 'Earnings', 'Perf Sem %', 'Perf Mes %',
           'SCTR', 'RSI(14)', 'ADX(14)', 'Beta', 'ATR(14)', 'EMA20']

try:
    # Buscar o crear la hoja
    try:
        ws = ss.worksheet(SHEET_WL_V5)
        ws.clear()  # Limpia TODO el contenido
    except gspread.WorksheetNotFound:
        ws = ss.add_worksheet(title=SHEET_WL_V5, rows=max(len(wl_data)+10, 100), cols=15)
    
    # Redimensionar si es necesario (para que entren todos los datos)
    current_rows = ws.row_count
    needed_rows = len(wl_data) + 5  # headers + buffer
    if current_rows < needed_rows:
        ws.resize(rows=needed_rows, cols=15)
    
    # Escribir headers + datos usando sintaxis moderna
    all_values = [headers] + wl_data
    ws.update(range_name='A1', values=all_values)
    
    # Formatear header
    ws.format('A1:M1', {
        'textFormat': {'bold': True},
        'backgroundColor': {'red': 0.1, 'green': 0.1, 'blue': 0.18}
    })
    
    print(f'✅ WL V5 escrita en Google Sheets: {len(wl_data)} tickers')
    print(f'   Hoja: {SHEET_WL_V5}')
    print(f'   Spreadsheet: {ss.title}')
    print(f'   Total filas escritas (incluye header): {len(all_values)}')
    
except Exception as e:
    print(f'❌ Error escribiendo en Sheets: {e}')
    print('   Verificá que estés autenticado (Paso 1).')

## Paso 8: Comparar con Radar V4 actual

Si ya tenés el Radar V4 generado, podemos ver cuántas coinciden.

In [ ]:
# Opcional: leer tickers del Radar V4 actual
try:
    radar_ws = ss.worksheet('🎯 Radar Semanal')
    radar_data = radar_ws.get_all_values()
    radar_tickers = set([r[0].upper() for r in radar_data[5:] if r])
    
    wl_v5_tickers = set([r[0].upper() for r in wl_data])
    coinciden = radar_tickers & wl_v5_tickers
    
    print(f'📊 Comparación con Radar V4:')
    print(f'   Radar V4: {len(radar_tickers)} tickers')
    print(f'   WL V5: {len(wl_v5_tickers)} tickers')
    print(f'   ✅ Coinciden: {len(coinciden)} tickers')
    print(f'   📌 Solo en V5: {len(wl_v5_tickers - radar_tickers)}')
    if coinciden:
        print(f'\n   Tickers coincidentes: {sorted(coinciden)[:20]}')
except Exception as e:
    print(f'⚠️ No se pudo comparar: {e}')

---

## ✅ Listo

Tu **WL V5 Generado** ya está en Google Sheets. Ahora podés:
1. Copiar esos tickers al **📋 WL CDI** (reemplazándolo o mezclando)
2. Ejecutar **🔄 Generar Radar Semanal** en tu Tracker V4
3. Comparar los resultados entre WL CDI (humano) y WL V5 (algoritmo)

> 💡 **Recomendación:** Corré esto cada **domingo** antes de la semana de trading.